# Oakland corridor activity — provider bridge and origin geography preflight

Run top to bottom in the same Snowflake environment as pilots 1–3. This fourth pilot reuses `TMP_EB_P2_CORRIDOR_GEOHASH9` and scans **only six individual stop dates**, four individual HOME dates, and two bounded HOME snapshot dates for Bay Area sample counts. It does not repeat a week or month extraction, and never exports device identifiers.

The questions are: (1) whether LIMA/SIERRA continue into the PAPA period, (2) whether historical corridor devices have same-provider HOME origins, (3) whether provider device IDs overlap on the same date, (4) whether the 2019 and current dotted CBG codes match appropriate Census geography, and (5) how long a full Bay Area CBG sample denominator takes. A missing 2019–2024 overlap or weak historical HOME coverage affects the meaning of a single normalized trend. Stop and record any source query that exceeds 10 minutes.

The geography check obtains California 2019 and 2020 block-group and place ZIPs from the [US Census TIGER/Line archive](https://www.census.gov/geographies/mapping-files/time-series/geo/tiger-line-file.2020.html) once (about 115 MB total). If downloads are unavailable, put these files under `./geo/census/` and rerun that section. No nationwide geometry is downloaded: this pilot validates the Bay Area origin categories and identifies the remaining national-distance requirement. The 2019 TIGER file carries pre-2020 geography, while the 2020 file provides the next vintage; **the match rate, not the filename alone, determines whether these candidate vintages fit the source codes**.

## 1. Connection and controls

In [1]:
import time
from pathlib import Path
import pandas as pd
from IPython.display import display

%load_ext cuebiqmagic.magics
%init_cuebiq_data
snow_engine=get_ipython().user_ns['instance']


schema : args.schema='paas_cda_pe_v3' and database: args.database='cuebiq_data' used
The connection to the Snowflake cluster has been established using database 'cuebiq_data' and schema 'paas_cda_pe_v3'.

You can now retrieve the connection using the following statement:
snow_engine = get_ipython().user_ns['instance']

The connection offers two methods


snow_engine.execute_statement("<SQL Statement>") # execute the SQL statement.


snow_engine.read_sql("<SQL Statement>") # execute the SQL statement and return a Pandas DataFrame.


snow_engine.write_dataframe(dataframe:pd.DataFrame, table_name:str,schema:str, index_in=False, if_exists_in="append",method_in="multi")


In [2]:
SOURCE_SCHEMA='CUEBIQ_DATA.PAAS_CDA_PE_V3'
STOP_TABLE=f'{SOURCE_SCHEMA}.STOP_BY_EVENT_DATE_UPLEVELLED'
HOME_TABLE=f'{SOURCE_SCHEMA}.DEVICE_RECURRING_AREA'
TMP_SCHEMA='DEDICATED.EDDIT_EAST_BAY_TMP'
COVER_TABLE=f'{TMP_SCHEMA}.TMP_EB_P2_CORRIDOR_GEOHASH9'
P=f'{TMP_SCHEMA}.TMP_EB_P4_'
COUNTRY_CODE='US'
# LIMA and SIERRA were in the 2019 corridor output; PAPA appeared by July 2023.
KNOWN_PROVIDERS=['LIMA','SIERRA','PAPA']
COHORT_DATES=[20190110,20230710,20240110,20250110]
BRIDGE_DATES=[20210110,20220110]
HOME_DATES=COHORT_DATES
DENOM_DATES=[20190110,20240110]
RUN_PROVIDER_COHORTS=True
RUN_HOME_MATCH=True
RUN_BAY_DENOMINATORS=True
RUN_CENSUS_GEOGRAPHY=True
AUTO_DOWNLOAD_CENSUS=True
CENSUS_DIR=Path('./geo/census')
EXPORT_DIR=Path('./activity/pilot4')
MAX_RAW_STOP_MINUTES=480
BAY_COUNTIES=['001','013','041','055','075','081','085','095','097']
EAST_BAY_COUNTIES=['001','013']
# No implicit national centroid assignment; Rest US can be categorized from state code.
print('Cohort dates:',COHORT_DATES,'Bay denominator dates:',DENOM_DATES)

Cohort dates: [20190110, 20230710, 20240110, 20250110] Bay denominator dates: [20190110, 20240110]


## 2. Helpers and existing cover check

In [3]:
timings=[]
results={}
def run(label,sql,source=False):
    print('Start:',label,flush=True)
    start=time.monotonic()
    try: out=snow_engine.read_sql(sql)
    except Exception:
        timings.append({'step':label,'seconds':round(time.monotonic()-start,1),
                        'source_scan':source,'status':'ERROR'})
        raise
    out.columns=[str(c).lower() for c in out.columns]
    elapsed=round(time.monotonic()-start,1)
    timings.append({'step':label,'seconds':elapsed,'source_scan':source,'status':'OK'})
    print(f'Done: {label} ({elapsed:.1f} s)',flush=True)
    return out

def quoted(values):
    return '('+','.join("'"+str(v).replace("'","''")+"'" for v in values)+')'

cover_check=run('Check persisted 11-corridor cover',f"""
SELECT COUNT(*) AS cover_rows,COUNT(DISTINCT corridor_id) AS corridors
FROM {COVER_TABLE}""")
display(cover_check)
if int(cover_check.iloc[0].cover_rows)!=218259 or int(cover_check.iloc[0].corridors)!=11:
    raise ValueError('Complete pilot 2 corridor cover is missing; do not run source scans.')
provider_sql=quoted(KNOWN_PROVIDERS)
county_sql=quoted(BAY_COUNTIES)

def cohort_table(day): return P+'COHORT_'+str(day)
def home_table(day): return P+'HOME_MATCH_'+str(day)

Start: Check persisted 11-corridor cover
Done: Check persisted 11-corridor cover (1.4 s)


,cover_rows,corridors
0,218259,11


## 3. Provider presence, corridor volumes, and ID overlap

Each stop query uses `country_code` and one exact `event_date`. It discovers all providers in the corridor geography, while the HOME and denominator checks below focus on the three providers already seen. The extra 2021/2022 dates check the historical bridge without rescanning HOME. A shared `cuebiq_id` across providers on a day is a warning against summing provider unique counts; no observed overlap would not prove that their physical-device panels are disjoint.

In [4]:
if RUN_PROVIDER_COHORTS:
    provider_frames=[];overlap_frames=[]
    for day in COHORT_DATES+BRIDGE_DATES:
        table=cohort_table(day)
        run(f'Persist corridor devices {day}',f"""
        CREATE OR REPLACE TABLE {table} AS
        SELECT s.provider_id,s.cuebiq_id,c.corridor_id,c.corridor_name,
               COUNT(*) AS matched_stop_slices
        FROM {STOP_TABLE} s JOIN {COVER_TABLE} c ON s.geohash_id=c.geohash9
        WHERE s.country_code='{COUNTRY_CODE}' AND s.event_date={day}
          AND s.stop_zoned_datetime IS NOT NULL
          AND s.dwell_time_minutes>0 AND s.dwell_time_minutes<={MAX_RAW_STOP_MINUTES}
        GROUP BY 1,2,3,4""",source=True)
        q=run(f'Provider activity {day}',f"""
        SELECT provider_id,COUNT(DISTINCT corridor_id) AS corridors,
               COUNT(DISTINCT cuebiq_id) AS unique_devices,
               SUM(matched_stop_slices) AS matched_stop_slices
        FROM {table} GROUP BY provider_id ORDER BY provider_id""")
        q.insert(0,'event_date',day)
        provider_frames.append(q)
        overlap=run(f'Cross-provider ID overlap {day}',f"""
        WITH p AS (SELECT DISTINCT provider_id,cuebiq_id FROM {table}),
        d AS (SELECT cuebiq_id,COUNT(*) AS provider_count FROM p GROUP BY 1)
        SELECT COUNT_IF(provider_count>=2) AS ids_seen_in_multiple_providers,
               COUNT(*) AS ids_seen_in_any_provider FROM d""")
        overlap.insert(0,'event_date',day)
        overlap_frames.append(overlap)
    results['provider_days']=pd.concat(provider_frames,ignore_index=True)
    results['id_overlap']=pd.concat(overlap_frames,ignore_index=True)
    display(results['provider_days'])
    display(results['id_overlap'])
    unseen=sorted(set(results['provider_days'].provider_id.dropna())-set(KNOWN_PROVIDERS))
    if unseen: print('Additional providers found; add these to KNOWN_PROVIDERS before HOME interpretation:',unseen)
else:
    print('Provider cohort queries skipped; HOME matching also requires them.')

Start: Persist corridor devices 20190110
Done: Persist corridor devices 20190110 (26.7 s)
Start: Provider activity 20190110
Done: Provider activity 20190110 (0.5 s)
Start: Cross-provider ID overlap 20190110
Done: Cross-provider ID overlap 20190110 (0.5 s)
Start: Persist corridor devices 20230710
Done: Persist corridor devices 20230710 (38.4 s)
Start: Provider activity 20230710
Done: Provider activity 20230710 (0.6 s)
Start: Cross-provider ID overlap 20230710
Done: Cross-provider ID overlap 20230710 (0.4 s)
Start: Persist corridor devices 20240110
Done: Persist corridor devices 20240110 (38.0 s)
Start: Provider activity 20240110
Done: Provider activity 20240110 (0.5 s)
Start: Cross-provider ID overlap 20240110
Done: Cross-provider ID overlap 20240110 (0.5 s)
Start: Persist corridor devices 20250110
Done: Persist corridor devices 20250110 (47.8 s)
Start: Provider activity 20250110
Done: Provider activity 20250110 (0.5 s)
Start: Cross-provider ID overlap 20250110
Done: Cross-provider ID o

,event_date,provider_id,corridors,unique_devices,matched_stop_slices
0,20190110,LIMA,11,2598,4335
1,20190110,SIERRA,11,2292,4044
2,20230710,PAPA,11,4061,5665
3,20230710,SIERRA,11,88,162
4,20230710,WHISKEY,3,5,5
5,20240110,PAPA,11,4286,6148
6,20240110,SIERRA,11,94,174
7,20240110,WHISKEY,7,14,14
8,20250110,PAPA,11,6269,8132
9,20250110,SIERRA,11,68,123


,event_date,ids_seen_in_multiple_providers,ids_seen_in_any_provider
0,20190110,21,4869
1,20230710,3,4151
2,20240110,0,4394
3,20250110,0,6350
4,20210110,2,1064
5,20220110,3,1965


Additional providers found; add these to KNOWN_PROVIDERS before HOME interpretation: ['WHISKEY']


## 4. Same-provider HOME quality for corridor visitors

A dated HOME snapshot is joined to distinct corridor devices **on provider and device ID**. The output compares match, confidence, dotted-code pattern, and CBG counts by provider without exporting device IDs. The 2019 rows are the critical historical feasibility check. Historical snapshots are not assumed to use 2024's CBG boundary vintage.

In [5]:
if RUN_PROVIDER_COHORTS and RUN_HOME_MATCH:
    quality=[];formats=[];examples=[]
    for day in HOME_DATES:
        cohort=cohort_table(day); target=home_table(day)
        run(f'Persist provider-matched HOME {day}',f"""
        CREATE OR REPLACE TABLE {target} AS
        WITH devices AS (
          SELECT DISTINCT provider_id,cuebiq_id FROM {cohort}
          WHERE provider_id IN {provider_sql}
        ), ranked AS (
          SELECT d.provider_id,d.cuebiq_id,h.block_group_id,h.confidence_level,
            ROW_NUMBER() OVER (PARTITION BY d.provider_id,d.cuebiq_id
              ORDER BY h.confidence_level DESC NULLS LAST,h.block_group_id) AS rn
          FROM devices d JOIN {HOME_TABLE} h
            ON h.provider_id=d.provider_id AND h.cuebiq_id=d.cuebiq_id
          WHERE h.provider_id IN {provider_sql} AND h.country_code='{COUNTRY_CODE}'
            AND h.snapshot_event_date={day} AND h.tag_type_code='HOME'
        )
        SELECT d.provider_id,d.cuebiq_id,r.block_group_id,r.confidence_level
        FROM devices d LEFT JOIN ranked r
          ON d.provider_id=r.provider_id AND d.cuebiq_id=r.cuebiq_id AND r.rn=1
        """,source=True)
        q=run(f'HOME match summary {day}',f"""
        SELECT provider_id,COUNT(*) AS corridor_devices,
          COUNT_IF(block_group_id IS NOT NULL AND TRIM(block_group_id)<>'') AS devices_with_cbg,
          COUNT_IF(confidence_level>=0.60 AND block_group_id IS NOT NULL
            AND TRIM(block_group_id)<>'') AS confident_cbg_devices,
          COUNT(DISTINCT block_group_id) AS distinct_cbg_values
        FROM {target} GROUP BY provider_id ORDER BY provider_id""")
        q.insert(0,'snapshot_date',day);quality.append(q)
        fmt=run(f'HOME code shapes {day}',f"""
        SELECT provider_id,LENGTH(TRIM(block_group_id)) AS code_length,
          COUNT(*) AS devices,COUNT_IF(confidence_level>=0.60) AS confident_devices
        FROM {target} WHERE block_group_id IS NOT NULL AND TRIM(block_group_id)<>''
        GROUP BY 1,2 ORDER BY 1,2""")
        fmt.insert(0,'snapshot_date',day);formats.append(fmt)
        ex=run(f'HOME code examples {day}',f"""
        WITH counts AS (
          SELECT provider_id,block_group_id,COUNT(*) AS devices
          FROM {target} WHERE block_group_id IS NOT NULL AND TRIM(block_group_id)<>''
          GROUP BY 1,2
        )
        SELECT provider_id,block_group_id,devices FROM counts
        QUALIFY ROW_NUMBER() OVER
          (PARTITION BY provider_id ORDER BY devices DESC,block_group_id)<=5
        ORDER BY provider_id,devices DESC""")
        ex.insert(0,'snapshot_date',day);examples.append(ex)
    results['home_match_by_provider']=pd.concat(quality,ignore_index=True)
    results['home_code_lengths']=pd.concat(formats,ignore_index=True)
    results['home_code_examples']=pd.concat(examples,ignore_index=True)
    display(results['home_match_by_provider'])
    display(results['home_code_lengths'])
    display(results['home_code_examples'])
else:
    print('HOME matching skipped.')

Start: Persist provider-matched HOME 20190110
Done: Persist provider-matched HOME 20190110 (17.7 s)
Start: HOME match summary 20190110
Done: HOME match summary 20190110 (0.5 s)
Start: HOME code shapes 20190110
Done: HOME code shapes 20190110 (0.4 s)
Start: HOME code examples 20190110
Done: HOME code examples 20190110 (0.5 s)
Start: Persist provider-matched HOME 20230710
Done: Persist provider-matched HOME 20230710 (129.8 s)
Start: HOME match summary 20230710
Done: HOME match summary 20230710 (0.5 s)
Start: HOME code shapes 20230710
Done: HOME code shapes 20230710 (0.4 s)
Start: HOME code examples 20230710
Done: HOME code examples 20230710 (0.4 s)
Start: Persist provider-matched HOME 20240110
Done: Persist provider-matched HOME 20240110 (161.2 s)
Start: HOME match summary 20240110
Done: HOME match summary 20240110 (0.5 s)
Start: HOME code shapes 20240110
Done: HOME code shapes 20240110 (0.6 s)
Start: HOME code examples 20240110
Done: HOME code examples 20240110 (0.4 s)
Start: Persist pr

,snapshot_date,provider_id,corridor_devices,devices_with_cbg,confident_cbg_devices,distinct_cbg_values
0,20190110,LIMA,2598,2582,2566,1355
1,20190110,SIERRA,2292,2237,2196,1246
2,20230710,PAPA,4061,3459,3012,1711
3,20230710,SIERRA,88,88,88,79
4,20240110,PAPA,4286,3610,3002,1690
5,20240110,SIERRA,94,92,91,85
6,20250110,PAPA,6269,5180,4094,2276
7,20250110,SIERRA,68,66,65,62


,snapshot_date,provider_id,code_length,devices,confident_devices
0,20190110,LIMA,18,2582,2566
1,20190110,SIERRA,18,2237,2196
2,20230710,PAPA,18,3459,3012
3,20230710,SIERRA,18,88,88
4,20240110,PAPA,18,3610,3002
5,20240110,SIERRA,18,92,91
6,20250110,PAPA,18,5180,4094
7,20250110,SIERRA,18,66,65


,snapshot_date,provider_id,block_group_id,devices
0,20190110,LIMA,US.CA.001.403000.1,12
1,20190110,LIMA,US.CA.001.401400.2,11
2,20190110,LIMA,US.CA.001.403400.2,11
3,20190110,LIMA,US.CA.001.400500.1,10
4,20190110,LIMA,US.CA.001.402800.2,10
5,20190110,SIERRA,US.CA.001.403300.1,24
6,20190110,SIERRA,US.CA.001.402800.2,23
7,20190110,SIERRA,US.CA.001.402900.1,20
8,20190110,SIERRA,US.CA.001.403100.1,19
9,20190110,SIERRA,US.CA.001.401300.3,15


## 5. Bay Area sample denominator by provider

This reads **one HOME snapshot partition per selected year**, grouping confident devices by dotted CBG code for all nine Bay Area counties, separately by provider. Unlike pilot 3's 25 visitor-selected CBGs, this is a region-wide sample-cost benchmark. The 2019 and 2024 provider counts are not themselves a longitudinal correction if the provider panels do not overlap. No national origin distance or all-US denominator is estimated here.

In [6]:
if RUN_BAY_DENOMINATORS:
    denoms=[]
    for day in DENOM_DATES:
        q=run(f'All-Bay CBG sample {day}',f"""
        SELECT provider_id,block_group_id,
               COUNT(DISTINCT cuebiq_id) AS sample_devices
        FROM {HOME_TABLE}
        WHERE provider_id IN {provider_sql} AND country_code='{COUNTRY_CODE}'
          AND snapshot_event_date={day} AND tag_type_code='HOME'
          AND confidence_level>=0.60
          AND SPLIT_PART(block_group_id,'.',1)='US'
          AND SPLIT_PART(block_group_id,'.',2)='CA'
          AND SPLIT_PART(block_group_id,'.',3) IN {county_sql}
        GROUP BY 1,2""",source=True)
        q.insert(0,'snapshot_date',day);denoms.append(q)
        print('CBG rows returned:',len(q),'per-provider CBGs and device totals:')
        display(q.groupby('provider_id',as_index=False).agg(
          cbgs=('block_group_id','nunique'),sum_cbg_devices=('sample_devices','sum')))
    results['bay_sample_by_cbg']=pd.concat(denoms,ignore_index=True)
else:
    print('Region-wide denominator cost remains untested.')

Start: All-Bay CBG sample 20190110
Done: All-Bay CBG sample 20190110 (12.9 s)
CBG rows returned: 9485 per-provider CBGs and device totals:


,provider_id,cbgs,sum_cbg_devices
0,LIMA,4738,159257
1,SIERRA,4747,172638


Start: All-Bay CBG sample 20240110
Done: All-Bay CBG sample 20240110 (97.2 s)
CBG rows returned: 8959 per-provider CBGs and device totals:


,provider_id,cbgs,sum_cbg_devices
0,PAPA,4755,2234689
1,SIERRA,4204,13934


## 6. Census geography files and block-group matching

This is a local geography check; no Snowflake source scan. Four official California TIGER/Line ZIPs are reused after their first download: 2019 and 2020 block groups, and corresponding places. For the Bay Area, match source dotted codes to Census block-group GEOIDs using California `06` + county (3) + tract (6) + block group (1). Compare 2019 codes against **both** vintages, likewise 2024 codes, before adopting a vintage. Categorize each Bay CBG using its point on surface and the Oakland place boundary of the same vintage; San Francisco is county `075`. The other California and US categories follow state/county codes. A point rule is a practical pilot choice for border-crossing CBGs and must be documented in the full run.

In [7]:
from urllib.request import urlretrieve
CENSUS_FILES={
 'bg_2019':(CENSUS_DIR/'tl_2019_06_bg.zip',
   'https://www2.census.gov/geo/tiger/TIGER2019/BG/tl_2019_06_bg.zip'),
 'place_2019':(CENSUS_DIR/'tl_2019_06_place.zip',
   'https://www2.census.gov/geo/tiger/TIGER2019/PLACE/tl_2019_06_place.zip'),
 'bg_2020':(CENSUS_DIR/'tl_2020_06_bg.zip',
   'https://www2.census.gov/geo/tiger/TIGER2020/BG/tl_2020_06_bg.zip'),
 'place_2020':(CENSUS_DIR/'tl_2020_06_place.zip',
   'https://www2.census.gov/geo/tiger/TIGER2020/PLACE/tl_2020_06_place.zip'),
}
missing=[]
if RUN_CENSUS_GEOGRAPHY:
    CENSUS_DIR.mkdir(parents=True,exist_ok=True)
    for key,(path,url) in CENSUS_FILES.items():
        if path.exists() and path.stat().st_size>100_000:
            print('Reusing',path)
            continue
        if not AUTO_DOWNLOAD_CENSUS:
            missing.append(str(path));continue
        print('Downloading',path.name,flush=True)
        try:
            urlretrieve(url,path)
            print('Saved',path.name,round(path.stat().st_size/1e6,1),'MB')
        except Exception as exc:
            path.unlink(missing_ok=True)
            missing.append(str(path))
            print('Download unavailable:',path.name,type(exc).__name__,str(exc)[:160])
    if missing: print('Geography check will be marked incomplete. Place these ZIPs:',missing)
else: print('Census geography check skipped.')

Saved tl_2019_06_bg.zip 50.0 MB
Saved tl_2019_06_place.zip 9.5 MB
Saved tl_2020_06_bg.zip 51.9 MB
Saved tl_2020_06_place.zip 9.6 MB


In [8]:
import re
code_pattern=re.compile(r'^US\.([A-Z]{2})\.(\d{3})\.(\d{6})\.(\d)$')
def parse_cbg(code):
    m=code_pattern.fullmatch(str(code))
    if not m: return (None,None,None,None)
    state,county,tract,bg=m.groups()
    return (state,county,'06'+county+tract+bg if state=='CA' else None,bg)

if RUN_CENSUS_GEOGRAPHY and not missing:
    import geopandas as gpd
    BBOX=(-124.0,36.5,-121.0,39.5)
    geo={}
    for vintage in (2019,2020):
        start=time.monotonic()
        bg=gpd.read_file(CENSUS_FILES[f'bg_{vintage}'][0],bbox=BBOX)
        place=gpd.read_file(CENSUS_FILES[f'place_{vintage}'][0],bbox=BBOX)
        geoid_col='GEOID' if 'GEOID' in bg else 'GEOID20'
        bg['geoid']=bg[geoid_col].astype(str)
        bg['county']=bg.geoid.str.slice(2,5)
        bg=bg.loc[bg.county.isin(BAY_COUNTIES),['geoid','county','geometry']].copy()
        oak=place.loc[place['NAME'].astype(str).eq('Oakland')]
        if len(oak)!=1: raise ValueError(f'Expected one Oakland place in {vintage} file, got {len(oak)}')
        oak_geometry=oak.to_crs('EPSG:3310').geometry.iloc[0]
        projected=bg.to_crs('EPSG:3310')
        inside=projected.geometry.representative_point().within(oak_geometry)
        bg['geography']='Rest Bay Area'
        bg.loc[bg.county.isin(EAST_BAY_COUNTIES),'geography']='Rest East Bay'
        bg.loc[bg.county.eq('075'),'geography']='San Francisco'
        bg.loc[inside.to_numpy(),'geography']='Oakland'
        if bg.geoid.duplicated().any(): raise ValueError('Duplicated CBG GEOID')
        geo[vintage]=bg[['geoid','county','geography']].copy()
        print(vintage,'Bay block groups:',len(bg),'Oakland:',inside.sum(),
              'load/classify seconds:',round(time.monotonic()-start,1))

    # Dated corridor HOME CBGs, with provider and visitor weights; no IDs.
    source=[]
    if 'home_match_by_provider' in results:
        for day in HOME_DATES:
            q=run(f'Grouped observed HOME codes {day}',f"""
            SELECT provider_id,block_group_id,COUNT(*) AS visitor_devices
            FROM {home_table(day)}
            WHERE block_group_id IS NOT NULL AND TRIM(block_group_id)<>''
            GROUP BY 1,2""")
            q.insert(0,'snapshot_date',day);source.append(q)
    if source:
        codes=pd.concat(source,ignore_index=True)
        parsed=codes.block_group_id.map(parse_cbg).tolist()
        codes[['state','county','geoid','block_group']]=pd.DataFrame(parsed,index=codes.index)
        qa=[]
        for day in HOME_DATES:
            subset=codes.loc[codes.snapshot_date.eq(day)].copy()
            for vintage in (2019,2020):
                merged=subset.merge(geo[vintage],on='geoid',how='left',suffixes=('','_census'))
                bay=merged.state.eq('CA') & merged.county.isin(BAY_COUNTIES)
                for provider,g in merged.groupby('provider_id'):
                    b=g.loc[bay.loc[g.index]]
                    qa.append({'snapshot_date':day,'provider_id':provider,'candidate_vintage':vintage,
                      'all_visitor_devices':int(g.visitor_devices.sum()),
                      'parsed_visitor_devices':int(g.loc[g.state.notna(),'visitor_devices'].sum()),
                      'bay_visitor_devices':int(b.visitor_devices.sum()),
                      'bay_matched_visitor_devices':int(b.loc[b.geography.notna(),'visitor_devices'].sum()),
                      'bay_distinct_cbgs':len(b),'bay_matched_cbgs':int(b.geography.notna().sum())})
        results['census_vintage_match']=pd.DataFrame(qa)
        display(results['census_vintage_match'])
        # Categorize observed origins with candidate vintage chosen by snapshot year.
        categorized=[]
        for day,g in codes.groupby('snapshot_date'):
            vintage=2019 if int(day)<20210101 else 2020
            m=g.merge(geo[vintage][['geoid','geography']],on='geoid',how='left')
            m.loc[m.state.ne('CA') & m.state.notna(),'geography']='Rest US'
            m.loc[m.state.eq('CA') & ~m.county.isin(BAY_COUNTIES),'geography']='Rest California'
            m['geography']=m.geography.fillna('UNMATCHED')
            categorized.append(m)
        results['observed_origin_categories']=pd.concat(categorized,ignore_index=True)
        report=results['observed_origin_categories'].groupby(
          ['snapshot_date','provider_id','geography'],as_index=False).visitor_devices.sum()
        results['origin_category_qa']=report
        display(report)
    if 'bay_sample_by_cbg' in results:
        region=[]
        for day,g in results['bay_sample_by_cbg'].groupby('snapshot_date'):
            vintage=2019 if int(day)<20210101 else 2020
            sample=g.copy()
            sample['geoid']=[parse_cbg(x)[2] for x in sample.block_group_id]
            m=sample.merge(geo[vintage][['geoid','geography']],on='geoid',how='left')
            m['geography']=m.geography.fillna('UNMATCHED')
            region.append(m.groupby(['snapshot_date','provider_id','geography'],as_index=False)
                          .agg(cbgs=('block_group_id','nunique'),
                               summed_cbg_devices=('sample_devices','sum')))
        results['regional_denominator_qa']=pd.concat(region,ignore_index=True)
        display(results['regional_denominator_qa'])
else:
    print('Census geography QA unavailable; provider/HOME/denominator results remain valid.')

2019 Bay block groups: 4756 Oakland: 336 load/classify seconds: 3.9
2020 Bay block groups: 5187 Oakland: 354 load/classify seconds: 4.0
Start: Grouped observed HOME codes 20190110
Done: Grouped observed HOME codes 20190110 (0.7 s)
Start: Grouped observed HOME codes 20230710
Done: Grouped observed HOME codes 20230710 (0.6 s)
Start: Grouped observed HOME codes 20240110
Done: Grouped observed HOME codes 20240110 (0.5 s)
Start: Grouped observed HOME codes 20250110
Done: Grouped observed HOME codes 20250110 (0.4 s)


,snapshot_date,provider_id,candidate_vintage,all_visitor_devices,parsed_visitor_devices,bay_visitor_devices,bay_matched_visitor_devices,bay_distinct_cbgs,bay_matched_cbgs
0,20190110,LIMA,2019,2582,2582,2342,2342,1141,1141
1,20190110,SIERRA,2019,2237,2237,2023,2023,1039,1039
2,20190110,LIMA,2020,2582,2582,2342,2080,1141,1010
3,20190110,SIERRA,2020,2237,2237,2023,1775,1039,911
4,20230710,PAPA,2019,3459,3459,2982,2982,1286,1286
5,20230710,SIERRA,2019,88,88,73,73,64,64
6,20230710,PAPA,2020,3459,3459,2982,2616,1286,1124
7,20230710,SIERRA,2020,88,88,73,65,64,58
8,20240110,PAPA,2019,3610,3610,3195,3195,1303,1303
9,20240110,SIERRA,2019,92,92,82,82,75,75


,snapshot_date,provider_id,geography,visitor_devices
0,20190110,LIMA,Oakland,926
1,20190110,LIMA,Rest Bay Area,166
2,20190110,LIMA,Rest California,177
3,20190110,LIMA,Rest East Bay,1173
4,20190110,LIMA,Rest US,63
5,20190110,LIMA,San Francisco,77
6,20190110,SIERRA,Oakland,985
7,20190110,SIERRA,Rest Bay Area,174
8,20190110,SIERRA,Rest California,148
9,20190110,SIERRA,Rest East Bay,779


,snapshot_date,provider_id,geography,cbgs,summed_cbg_devices
0,20190110,LIMA,Oakland,334,5151
1,20190110,LIMA,Rest Bay Area,2482,85075
2,20190110,LIMA,Rest East Bay,1344,55941
3,20190110,LIMA,San Francisco,578,13090
4,20190110,SIERRA,Oakland,335,9409
5,20190110,SIERRA,Rest Bay Area,2486,92520
6,20190110,SIERRA,Rest East Bay,1347,55507
7,20190110,SIERRA,San Francisco,579,15202
8,20240110,PAPA,Oakland,326,117694
9,20240110,PAPA,Rest Bay Area,2113,976224


## 7. Grouped exports and decision checklist

The Bay denominator sums are **CBG-level device sums on one snapshot**, not a distinct Bay-wide device count if a device has multiple HOME assignments. The full method must pick one HOME per provider/device/snapshot before grouping. The provider bridge and 2019 HOME match determine whether a common 2019-normalized trend is defensible; otherwise separate provider series or an explicit calibrated bridge is needed. Do not sum LIMA and SIERRA devices merely because both have 2019 data. Capture query-profile bytes/partitions/credits in Snowflake for the dated HOME scans and the two region denominators.

In [9]:
results['timings']=pd.DataFrame(timings)
status=pd.DataFrame([
 {'gate':'Provider overlap 2019–2025','result':'MEASURED' if 'provider_days' in results else 'NOT_RUN'},
 {'gate':'Same-provider HOME on corridor devices','result':'MEASURED' if 'home_match_by_provider' in results else 'NOT_RUN'},
 {'gate':'Bay-wide sample denominator cost','result':'MEASURED' if 'bay_sample_by_cbg' in results else 'NOT_RUN'},
 {'gate':'2019 vs 2020 CBG geography match','result':'MEASURED' if 'census_vintage_match' in results else 'NEEDS_CENSUS_ZIPS'},
 {'gate':'National HOME distance centroids','result':'NOT_TESTED_IN_THIS_PILOT'},
])
results['decision_checklist']=status
EXPORT_DIR.mkdir(parents=True,exist_ok=True)
for name,frame in results.items():
    frame.to_csv(EXPORT_DIR/f'{name}.csv',index=False)
display(results['timings'])
display(status)
print('Grouped CSVs:',EXPORT_DIR.resolve())

,step,seconds,source_scan,status
0,Check persisted 11-corridor cover,1.4,False,OK
1,Persist corridor devices 20190110,26.7,True,OK
2,Provider activity 20190110,0.5,False,OK
3,Cross-provider ID overlap 20190110,0.5,False,OK
4,Persist corridor devices 20230710,38.4,True,OK
5,Provider activity 20230710,0.6,False,OK
6,Cross-provider ID overlap 20230710,0.4,False,OK
7,Persist corridor devices 20240110,38.0,True,OK
8,Provider activity 20240110,0.5,False,OK
9,Cross-provider ID overlap 20240110,0.5,False,OK


,gate,result
0,Provider overlap 2019–2025,MEASURED
1,Same-provider HOME on corridor devices,MEASURED
2,Bay-wide sample denominator cost,MEASURED
3,2019 vs 2020 CBG geography match,MEASURED
4,National HOME distance centroids,NOT_TESTED_IN_THIS_PILOT


Grouped CSVs: /home/jovyan/eddit-east-bay/activity/pilot4
